# Module 23: Interactive Strict Typing & Modern Packaging

### What You Will Discover
By running this notebook, you will explore static type analysis (`mypy --strict`), structural subtyping with `Protocols` (duck typing with static types), preserving wrapper signatures with `ParamSpec`, and modern `pyproject.toml` manifests.

**Key Question Answered:** *Why does `list[Dog]` fail to satisfy `list[Animal]` in static type checkers, and how do covariant immutable collections resolve it?*


In [ ]:
# Step 1: Structural subtyping with Protocol (PEP 544)
from typing import Protocol, runtime_checkable


@runtime_checkable
class Serializable(Protocol):
    def to_json(self) -> str: ...

class CustomerRecord:
    def __init__(self, name: str): self.name = name
    def to_json(self) -> str:
        return f'{{"name": "{self.name}"}}'


In [ ]:
# Step 2: Protocol verification without nominal inheritance
c = CustomerRecord('Ada Lovelace')
print(f'Does CustomerRecord inherit from Serializable? {issubclass(CustomerRecord, Serializable)}')
print(f'Is instance of Serializable? {isinstance(c, Serializable)}')
print('Explanation: Protocols enable structural subtyping statically and dynamically!')


In [ ]:
# Step 3: Generic TypeVar and Container types
from collections.abc import Sequence
from typing import Generic, TypeVar

T = TypeVar('T')
class Stack(Generic[T]):
    def __init__(self): self._items = []
    def push(self, item: T): self._items.append(item)
    def pop(self) -> T: return self._items.pop()


### 🔮 Prediction Prompt
**Before running the next cell:** In Python type systems, if `class Dog(Animal): pass`, is `list[Dog]` a subtype of `list[Animal]`? What about `Sequence[Dog]` vs `Sequence[Animal]`? Write down your prediction.


In [ ]:
# Surprising Result: Container Invariance vs Covariance
print('list[T] is INVARIANT: list[Dog] is NOT a subtype of list[Animal]!')
print('Reason: If a function accepted list[Animal] and appended a Cat, your list[Dog] is corrupted!')
print('Sequence[T] is COVARIANT: Sequence[Dog] IS a subtype of Sequence[Animal] because it is read-only!')


### Preserving Call Signatures with `ParamSpec`
`ParamSpec` captures parameter types and returns of wrapped functions for IDE autocomplete.


In [ ]:
import functools
from collections.abc import Callable
from typing import ParamSpec

P = ParamSpec('P')
R = TypeVar('R')

def audit_log(func: Callable[P, R]) -> Callable[P, R]:
    @functools.wraps(func)
    def wrapper(*args: P.args, **kwargs: P.kwargs) -> R:
        return func(*args, **kwargs)
    return wrapper

@audit_log
def send_email(to: str, subject: str, priority: int = 1) -> bool: return True
print('Decorated with ParamSpec signature preservation.')


### Modern `pyproject.toml` (PEP 517 / 621)
Standard declarative project specification without legacy `setup.py`.


In [ ]:
toml_spec = '''
[build-system]
requires = ["hatchling"]
build-backend = "hatchling.build"

[project]
name = "enterprise-sdk"
version = "1.0.0"
dependencies = ["httpx>=0.27.0", "pydantic>=2.6.0"]
'''
print(toml_spec)


### 🛠️ Interactive Challenge: Fix Incompatible Type Annotations
The following function accepts an immutable sequence of items but is annotated with mutable `list[str]`, causing invariant type errors when passing tuples. Fix the type annotation to `Sequence[str]`.


In [ ]:
# TODO: FIX ME - Change list[str] to Sequence[str] to accept tuples and lists covariantly

# FIX: def format_tags(tags: Sequence[str]) -> str:
def format_tags(tags: Sequence[str]) -> str:
    return ', '.join(tags)

tup_tags = ('security', 'production', 'v2')
print(f'Formatted: {format_tags(tup_tags)}')


### 🏁 Summary & Next Steps
- Use `Protocols` for structural duck-typing.
- Use `Sequence[T]` (covariant) for input collections; `list[T]` is invariant.
- Use `ParamSpec` to preserve higher-order function signatures.
- Run `python 01_generics_and_protocols_demo.py` and review `02_pyproject_toml_packaging_demo.md`.
- Complete [PROJECT_GUIDE.md](PROJECT_GUIDE.md) to build the typed SDK.
